In [3]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"


In [6]:
Artist_name ="Käthe Kollwitz"

In [7]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx")

In [8]:
claude_label.shape

(1000, 10)

In [9]:
sum(claude_label['artwork id'].duplicated())

0

In [23]:
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_{Artist_name.split(" ")[-1]}.xlsx")

In [24]:
gemini_label.shape

(1000, 10)

In [25]:
sum(gemini_label['artwork id'].duplicated())

0

In [26]:
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx")

In [27]:
openai_label.shape

(1000, 10)

In [28]:
sum(openai_label['artwork id'].duplicated())

0

In [29]:
full_df = claude_label.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_openai"))

In [30]:
full_df.shape

(1000, 14)

In [31]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_openai,artistic_value_comment_openai,creativity_answer_openai,creativity_comment_openai
0,424044055,Städtisches Obdach,Käthe,Kollwitz,1926,German,High,Kollwitz created this lithograph of an exhaust...,Yes,Kollwitz's experience as a mother was central ...,High,"Käthe Kollwitz's lithograph ""Städtisches Obdac...",Yes,"""Städtisches Obdach"" demonstrates Kollwitz's i..."
1,424049391,"Ein Weberaufstand (portfolio of 6, incl. 3 lit...",Käthe,Kollwitz,1897,German,High,Käthe Kollwitz achieved her first great succes...,Yes,The portfolio exhibits genuine creative innova...,High,"Käthe Kollwitz's ""A Weavers' Revolt"" (1893–189...",Yes,"Kollwitz's ""A Weavers' Revolt"" stands out for ..."
2,424050836,Ruf des Todes pl.8 (from Tod),Käthe,Kollwitz,1934,German,High,This work serves as Kollwitz's final print cyc...,Yes,Death was one of Kollwitz's most persistent th...,High,"""Ruf des Todes"" (Call of Death), created by Kä...",Yes,"Käthe Kollwitz's ""Ruf des Todes"" demonstrates ..."
3,424050837,Mutter mit Jungen,Käthe,Kollwitz,1931,German,High,The relationship between mother and child was ...,Yes,Kollwitz's representations avoid sentimentalit...,High,"Käthe Kollwitz's ""Mutter mit Jungen"" (Mother w...",Yes,"Kollwitz's ""Mutter mit Jungen"" demonstrates cr..."
4,424057623,Schwangere Frau,Käthe,Kollwitz,1910,German,High,Käthe Kollwitz is regarded as probably the mos...,Yes,While influenced by Berlin's 19th-century draw...,High,"Käthe Kollwitz's 1910 etching, ""Schwangere Fra...",Yes,"""Schwangere Frau"" demonstrates Kollwitz's inno..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,426269146,Ein Weberaufstand (suite of 6),Käthe,Kollwitz,1897,German,High,The graphic cycle Ein Weberaufstand achieved K...,Yes,Directly inspired by Hauptmann's drama Die Web...,High,"Käthe Kollwitz's ""Ein Weberaufstand"" (A Weaver...",Yes,"Kollwitz's ""Ein Weberaufstand"" is a testament ..."
996,426269149,Tod und Frau um das Kind ringend,Käthe,Kollwitz,1911,German,High,Käthe Kollwitz was a German artist whose Expre...,Yes,This work demonstrates genuine creativity thro...,High,"Käthe Kollwitz's 1911 etching, ""Tod und Frau u...",Yes,"Kollwitz's ""Tod und Frau um das Kind ringend"" ..."
997,426269150,Pflugzieher und Weib,Käthe,Kollwitz,1902,German,High,Käthe Kollwitz was a German artist whose Expre...,Yes,The work demonstrates creativity through its i...,High,"""Pflugzieher und Weib"" (Plough-Puller and Wife...",Yes,"Kollwitz's ""Pflugzieher und Weib"" showcases si..."
998,426269152,"Gefangene, Musik hörend",Käthe,Kollwitz,1925,German,High,Kollwitz was a German artist whose Expressioni...,Yes,"""It is my duty to voice the sufferings of huma...",High,"""Gefangene, Musik hörend"" (Prisoners Listening...",Yes,"Kollwitz's ""Gefangene, Musik hörend"" demonstra..."


In [32]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [33]:
full_df.shape

(1000, 18)

# Embedding Convert

In [34]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [36]:
def convert_one_row(model,i,texts,device):
    try:
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        text_embeds = outputs.pooler_output
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
        text_embeds=text_embeds.cpu().numpy()
    except Exception as e:
        print(f"Error processing {i}: {e}")
        text_embeds = np.zeros([2,512])
    return i, text_embeds

In [45]:
dataset = "claude"
if dataset =="claude":
    df = claude_label.copy()
elif dataset =="gemini":
    df = gemini_label.copy()
elif dataset =="openai":
    df = openai_label.copy()

In [46]:
number_size=df.shape[0]
#number_size=10
# range_start = 30000
range_start = 0
range_end = min(range_start+number_size,df.shape[0])
N = min(number_size, df.shape[0]-range_start)

In [47]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, 
                  [df.iloc[i].artistic_value_comment,df.iloc[i].creativity_comment],
                  device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, text_embeds = fut.result()
        embeddings[i-range_start] =text_embeds
        if i % 1000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-16 04:44:52: Start
2025-12-16 04:44:53: 0
Error processing 824: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
2025-12-16 04:45:06: Ends


In [48]:
np.save(f"clip_embeddings_{dataset}_{Artist_name.split(" ")[-1]}.npy", embeddings)

# Comment Check Consistency Rough

In [152]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini.strip()):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini.strip()):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [164]:
np.sum(consistent_artist)

np.int64(78)

In [163]:
np.sum(consistent_creative)

np.int64(73)

In [153]:
np.sum(consistent_overall)

np.int64(73)

In [154]:
check=full_df.copy()
check["creative_consist"]=consistent_creative
check["artistic_consist"]=consistent_artist
check["overall_consist"]=consistent_overall

# Comment Check Consistency Hard

In [155]:
claude_embed = np.load(f"clip_embeddings_claude.npy",allow_pickle=True)

In [156]:
gemini_embed = np.load(f"clip_embeddings_gemini.npy",allow_pickle=True)

In [157]:
openai_embed = np.load(f"clip_embeddings_openai.npy",allow_pickle=True)

In [158]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > 0.70).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > 0.70).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")

2025-11-28 08:04:09: Start
2025-11-28 08:04:09: Currently at 0


In [159]:
np.sum(embed_consistent_overall)

np.int64(20)

In [165]:
np.sum(embed_consistent_creative)

np.int64(31)

In [160]:
check["embed_artistic_consist"]=embed_consistent_artistic
check["embed_creative_consist"]=embed_consistent_creative
check["embed_overall_consist"]=embed_consistent_overall